In [169]:
from collections import defaultdict

import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from wordfreq import zipf_frequency
import numpy as np
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator, RegressorMixin
from catboost import CatBoostRegressor
from sklearn.model_selection import ParameterSampler

# 1. Feature Engineering

In [170]:

df = pd.read_csv("data/filtered.csv", index_col=0)
df.head()

,date,contest_num,word,n_reported,n_hard,try_1,try_2,try_3,try_4,try_5,try_6,try_x,hard/reported%,date_num
234,2022-05-11,326,farce,79446,5688,0,9,26,32,21,9,1,7.159580,0
233,2022-05-12,327,slung,75673,5419,0,2,16,37,31,13,2,7.161075,1
232,2022-05-13,328,tipsy,77585,5522,0,6,33,38,17,5,1,7.117355,2
231,2022-05-14,329,metal,73225,5290,1,10,31,34,18,7,1,7.224309,3
230,2022-05-15,330,yield,67115,4963,0,4,16,29,29,18,4,7.394770,4


In [171]:
VOWELS = set("aeiou")
RARE_LETTERS = set("zxqjkvbpyg")

In [172]:
# https://mathcenter.oxford.emory.edu/site/math125/englishLetterFreqs

ENGLISH_LETTER_FREQ = {
    'a': 0.0817, 'b': 0.0149, 'c': 0.0278, 'd': 0.0425, 'e': 0.1270,
    'f': 0.0223, 'g': 0.0202, 'h': 0.0609, 'i': 0.0697, 'j': 0.0015,
    'k': 0.0077, 'l': 0.0403, 'm': 0.0241, 'n': 0.0675, 'o': 0.0751,
    'p': 0.0193, 'q': 0.0010, 'r': 0.0599, 's': 0.0633, 't': 0.0906,
    'u': 0.0276, 'v': 0.0098, 'w': 0.0236, 'x': 0.0015, 'y': 0.0197,
    'z': 0.0007
}

START_FREQ = {
    't': 0.1594, 'a': 0.155, 'i': 0.0823, 's': 0.0775, 'o': 0.0712,
    'c': 0.0597, 'm': 0.0426, 'f': 0.0408, 'p': 0.040, 'w': 0.0382
}

END_FREQ = {
    'e': 0.1917, 's': 0.1435, 'd': 0.0923, 't': 0.0864, 'n': 0.0786,
    'y': 0.0730, 'r': 0.0693, 'o': 0.0467, 'l': 0.0456, 'f': 0.0408
}

COMMON_BIGRAMS = {
    "th","he","in","en","nt","re","er","an","ti","es",
    "on","at","se","nd","or","ar","al","te","co","de"
}

COMMON_TRIGRAMS = {
    "the","and","tha","ent","ing","ion","tio","for","nde"
}

In [173]:
def get_rarity(word):
    """higher = more common, lower = rarer"""
    freq = zipf_frequency(str(word).lower(), 'en')
    return max(0, 7 - freq)

In [174]:
def build_position_freq(words):
    position_counts = defaultdict(lambda: defaultdict(int))
    position_totals = defaultdict(int)

    for w in words:
        w = str(w).lower()
        for i, c in enumerate(w):
            position_counts[i][c] += 1
            position_totals[i] += 1

    position_freq = {}

    for i in position_counts:
        for c in position_counts[i]:
            position_freq[(i, c)] = (
                position_counts[i][c] / position_totals[i]
            )

    return position_freq


POSITION_FREQ = build_position_freq(df["word"])

In [175]:
def build_features(df):
    df = df.copy()

    df["word"] = df["word"].apply(lambda w: str(w).lower())

    df["rarity"] = df["word"].apply(get_rarity)

    # structural (KEEP)
    df["n_unique_letters"] = df["word"].apply(lambda x: len(set(x)))
    df["duplicate_letter_penalty"] = df["word"].apply(lambda x: len(x) - len(set(x)))
    df["n_vowels"] = df["word"].apply(lambda x: sum(c in VOWELS for c in x))
    df["n_rare_letters"] = df["word"].apply(lambda x: sum(c in RARE_LETTERS for c in x))

    # frequency (KEEP only 2, not 3)
    df["log_letter_score"] = df["word"].apply(
        lambda x: sum(np.log(max(ENGLISH_LETTER_FREQ.get(c, 1e-6), 1e-6)) for c in x)
    )
    df["rarity_weighted"] = df["word"].apply(
        lambda x: sum(1 / max(ENGLISH_LETTER_FREQ.get(c, 1e-6), 1e-6) for c in x)
    )

    # sequence (KEEP ALL — this is important)
    df["start_letter_score"] = df["word"].apply(lambda x: START_FREQ.get(x[0], 0) if x else 0)
    df["end_letter_score"] = df["word"].apply(lambda x: END_FREQ.get(x[-1], 0) if x else 0)

    df["bigram_score"] = df["word"].apply(
        lambda x: sum(1 for i in range(len(x) - 1) if x[i:i+2] in COMMON_BIGRAMS)
    )

    df["trigram_score"] = df["word"].apply(
        lambda x: sum(1 for i in range(len(x) - 2) if x[i:i+3] in COMMON_TRIGRAMS)
    )

    df["positional_entropy"] = df["word"].apply(
        lambda x: sum(
            -np.log(max(POSITION_FREQ.get((i, c), 1e-6), 1e-6))
            for i, c in enumerate(x)
        )
    )

    return df

# 2. Data preparation

In [176]:
TRY_COLS = [
    "try_1",
    "try_2",
    "try_3",
    "try_4",
    "try_5",
    "try_6",
    "try_x"
]

df = build_features(df)

FEATURE_NAMES = [
    "rarity",
    "n_unique_letters",
    "n_vowels",
    "n_rare_letters",
    "log_letter_score",
    "start_letter_score",
    "end_letter_score",
    "bigram_score",
    "trigram_score",
    "rarity_weighted",
    "positional_entropy",
    "duplicate_letter_penalty",
    "n_reported"
]

In [177]:
# Try cols sum = 1
df[TRY_COLS] = df[TRY_COLS].div(df[TRY_COLS].sum(axis=1), axis=0)

X = df[FEATURE_NAMES]

Y = df[TRY_COLS]

# 3. Model training

## Crterion

In [178]:
def kl_divergence(p, q):
    eps = 1e-9
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)
    return np.sum(p * np.log(p / q), axis=1)

In [179]:
def train_and_eval(model_generator, X_train, X_test, Y_train, Y_test, features):
    models = {}
    
    X_tr = X_train[features].values
    X_te = X_test[features].values
    
    for col in TRY_COLS:
        model = model_generator()
        model.fit(X_tr, Y_train[col])
        models[col] = model

    def predict_matrix(X):
        preds = np.column_stack([
            models[col].predict(X)
            for col in TRY_COLS
        ])
        preds = np.clip(preds, 0, None)
        preds = preds / preds.sum(axis=1, keepdims=True)
        return preds

    Y_pred = predict_matrix(X_te)
    Y_true_local = Y_test.values

    kl = kl_divergence(Y_true_local, Y_pred)
    return kl.mean()


## 3.2 Best feature combination search functions

In [180]:
def train_and_eval_kfold(
    model_generator,
    X_df,
    Y_df,
    features,
    n_splits=5,
    shuffle=True,
    random_state=42,
    mode="multi_model"  # "multi_model" or "single_model"
):
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)

    X = X_df[features].values
    Y = Y_df[TRY_COLS].values

    kl_scores = []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        Y_tr, Y_val = Y[train_idx], Y[val_idx]
        if mode == "multi_model":
            models = {}

            for i, col in enumerate(TRY_COLS):
                model = model_generator()
                model.fit(X_tr, Y_tr[:, i])
                models[col] = model

            def predict(X):
                preds = np.column_stack([
                    models[col].predict(X)
                    for col in TRY_COLS
                ])
                return preds

        elif mode == "single_model":
            model = model_generator()
            model.fit(X_tr, Y_tr)

            def predict(X):
                return model.predict(X)

        else:
            raise ValueError("mode must be 'multi_model' or 'single_model'")

        Y_pred = predict(X_val)

        # normalize to probability distribution per row
        Y_pred = np.clip(Y_pred, 0, None)
        Y_pred = Y_pred / (Y_pred.sum(axis=1, keepdims=True) + 1e-12)

        kl = kl_divergence(Y_val, Y_pred)
        kl_scores.append(kl.mean())

    return np.mean(kl_scores)

In [181]:
def forward_selection(model_generator, X, Y, n_splits=10, mode='multi_model'):
    remaining = set(FEATURE_NAMES)
    selected = []
    
    best_score = float("inf")

    while remaining:
        best_feature = None
        best_feature_score = float("inf")

        for feature in remaining:
            trial_features = selected + [feature]
            score = train_and_eval_kfold(
                model_generator,
                X,
                Y,
                trial_features,
                n_splits=n_splits,
                mode=mode
            )

            if score < best_feature_score:
                best_feature_score = score
                best_feature = feature

        # stop if no meaningful improvement
        if best_feature_score >= best_score - 1e-4:
            break

        selected.append(best_feature)
        remaining.remove(best_feature)
        best_score = best_feature_score

        print(f"Added: {best_feature} -> KL: {best_score:.6f}")

    print("\nFINAL BEST:")
    print("Features:", selected)
    print("Mean KL:", best_score)

    return selected, best_score

## 3.3 Linear regressions (one per number of tries)

In [182]:
linear_model_generator = lambda: make_pipeline(StandardScaler(), Ridge(150))
forward_selection(linear_model_generator, X, Y, mode="multi_model")

Added: n_unique_letters -> KL: 0.069749
Added: n_rare_letters -> KL: 0.064634
Added: rarity -> KL: 0.061941
Added: positional_entropy -> KL: 0.059480
Added: start_letter_score -> KL: 0.057724

FINAL BEST:
Features: ['n_unique_letters', 'n_rare_letters', 'rarity', 'positional_entropy', 'start_letter_score']
Mean KL: 0.057723866035749125


(['n_unique_letters',
  'n_rare_letters',
  'rarity',
  'positional_entropy',
  'start_letter_score'],
 np.float64(0.057723866035749125))

## 3.3 Softmax linear regressions

## 3.3.2 Finding the best feature combination

In [183]:
def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp = np.exp(x)
    return exp / exp.sum(axis=1, keepdims=True)

class SoftmaxModel(BaseEstimator, RegressorMixin):
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, Y):
        self.scaler = StandardScaler()
        Xs = self.scaler.fit_transform(X)

        # one linear model per class (safe, fixed size)

        self.models = []
        for i in range(Y.shape[1]):
            m = Ridge(alpha=self.alpha)
            m.fit(Xs, Y[:, i])
            self.models.append(m)

        return self

    def predict(self, X):
        Xs = self.scaler.transform(X)

        logits = np.column_stack([
            m.predict(Xs)
            for m in self.models
        ])

        return softmax(logits)

In [184]:
softmax_model_generator = lambda: SoftmaxModel(alpha=5)
forward_selection(softmax_model_generator, X, Y, mode="single_model")

Added: n_unique_letters -> KL: 0.369158
Added: log_letter_score -> KL: 0.366733
Added: rarity -> KL: 0.366224
Added: start_letter_score -> KL: 0.365847
Added: positional_entropy -> KL: 0.365635
Added: n_rare_letters -> KL: 0.365482

FINAL BEST:
Features: ['n_unique_letters', 'log_letter_score', 'rarity', 'start_letter_score', 'positional_entropy', 'n_rare_letters']
Mean KL: 0.36548163936009914


(['n_unique_letters',
  'log_letter_score',
  'rarity',
  'start_letter_score',
  'positional_entropy',
  'n_rare_letters'],
 np.float64(0.36548163936009914))

## 3.4 CatBoost

In [185]:
def run_catboost_feature_selection_grid(X, Y):
    param_dist = {
        "iterations": [100, 150, 200, 250],
        "depth": [3, 4, 5],
        "learning_rate": np.linspace(0.02, 0.08, 10),
        "l2_leaf_reg": [1, 3, 5, 7, 10]
    }

    sampled_params = list(ParameterSampler(param_dist, n_iter=20, random_state=42))

    best_global_score = float("inf")
    best_global_features = None
    best_global_params = None

    for params in sampled_params:

        print("\n==============================")
        print("Testing params:", params)
        print("==============================")

        def model_generator():
            return CatBoostRegressor(
                iterations=int(params["iterations"]),
                depth=int(params["depth"]),
                learning_rate=float(params["learning_rate"]),
                l2_leaf_reg=int(params["l2_leaf_reg"]),
                loss_function="MultiRMSE",
                random_seed=42,
                verbose=False
            )

        features, score = forward_selection(
            model_generator,
            X,
            Y,
            mode="single_model"
        )

        print("Result score:", score)
        print("Features:", features)

        if score < best_global_score:
            best_global_score = score
            best_global_features = features
            best_global_params = params

    print("\n\nFINAL BEST CONFIG (CATBOOST)")
    print("Score:", best_global_score)
    print("Features:", best_global_features)
    print("Params:", best_global_params)

    return best_global_features, best_global_params, best_global_score

In [186]:
cat_best_global_features, cat_best_global_params, cat_best_global_score = run_catboost_feature_selection_grid(X, Y)


Testing params: {'learning_rate': np.float64(0.02), 'l2_leaf_reg': 3, 'iterations': 200, 'depth': 3}
Added: n_unique_letters -> KL: 0.067743
Added: rarity_weighted -> KL: 0.057146
Added: rarity -> KL: 0.053293
Added: start_letter_score -> KL: 0.053146

FINAL BEST:
Features: ['n_unique_letters', 'rarity_weighted', 'rarity', 'start_letter_score']
Mean KL: 0.05314628898677619
Result score: 0.05314628898677619
Features: ['n_unique_letters', 'rarity_weighted', 'rarity', 'start_letter_score']

Testing params: {'learning_rate': np.float64(0.08), 'l2_leaf_reg': 3, 'iterations': 100, 'depth': 5}
Added: duplicate_letter_penalty -> KL: 0.067846
Added: log_letter_score -> KL: 0.058626
Added: rarity -> KL: 0.056525
Added: n_vowels -> KL: 0.054268
Added: rarity_weighted -> KL: 0.052700

FINAL BEST:
Features: ['duplicate_letter_penalty', 'log_letter_score', 'rarity', 'n_vowels', 'rarity_weighted']
Mean KL: 0.05269957715669014
Result score: 0.05269957715669014
Features: ['duplicate_letter_penalty', '

# 4. Predict for the word 'eerie' using the best model

In [194]:
def train_final_and_predict(X, Y, best_features, best_params, X_test):
    X_full = X[best_features].values
    Y_full = Y[TRY_COLS].values

    model = CatBoostRegressor(
        iterations=best_params["iterations"],
        learning_rate=best_params["learning_rate"],
        depth=best_params["depth"],
        l2_leaf_reg=best_params["l2_leaf_reg"],
        loss_function="MultiRMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(X_full, Y_full)

    X_test_feat = build_features(X_test.copy())
    X_test_feat = X_test_feat[best_features].values

    preds = model.predict(X_test_feat)

    preds = np.clip(preds, 0, None)
    preds = preds / preds.sum(axis=1, keepdims=True)

    for i, w in enumerate(X_test["word"]):
        print(f"\nPrediction for '{w}':")
        for col, p in zip(TRY_COLS, preds[i]):
            print(f"{col}: {p:.0%}")

    return preds

In [195]:
df_eerie = pd.DataFrame({"word": ["eerie"]})

train_final_and_predict(X, Y, cat_best_global_features, cat_best_global_params, df_eerie)


Prediction for 'eerie':
try_1: 0%
try_2: 5%
try_3: 18%
try_4: 32%
try_5: 29%
try_6: 13%
try_x: 2%


array([[0.00122201, 0.0498757 , 0.1840276 , 0.32036791, 0.29399888,
        0.13297247, 0.01753543]])